# CARLA360 — Dataset statistics & visualization
این نوت‌بوک مسیر دوربین، فاصلهٔ تصاویر، تغییر جهت و دامنهٔ زاویه‌ها را بررسی می‌کند.
محاسبات با پوزهای Standard_GT انجام می‌شوند و به GPU نیاز ندارند.

ابتدا در ترمینال پروژه اجرا کنید: `python -m pip install -r requirements-analysis.txt`
سپس همان محیط پایتون را به‌عنوان Kernel انتخاب کنید و سلول‌ها را به‌ترتیب اجرا کنید.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'analysis/dataset_stats.py').is_file(), 'Open this notebook inside carla360-benchmark'
sys.path.insert(0, str(ROOT))
from analysis.dataset_stats import analyze_sequence, comparison_row, export_statistics
from analysis.plots import make_figures, interactive_trajectory

# Change this path for another sequence. Linux paths are case-sensitive.
SEQUENCE = ROOT / 'standard_GT/city01_route01'
if not SEQUENCE.exists() and (ROOT / 'Standard_GT/city01_route01').exists():
    SEQUENCE = ROOT / 'Standard_GT/city01_route01'
TRANSLATION_THRESHOLD_M = 0.01  # 1 cm: a reporting threshold, not measurement accuracy.
ROTATION_THRESHOLD_DEG = 0.1


## Load poses and summarize
تمام فریم‌ها بررسی می‌شوند؛ لازم نیست برای محاسبهٔ آمار، تمام عکس‌ها وارد حافظه شوند.
فاصله و تغییر چرخش بین دو تصویر متوالیِ موجود محاسبه می‌شود. اگر فریم‌هایی حذف شده
باشند، این جفت ممکن است در تصویربرداری اصلی متوالی نباشد؛ جدول شکاف‌ها آن را نشان می‌دهد.


In [ ]:
result = analyze_sequence(SEQUENCE, TRANSLATION_THRESHOLD_M, ROTATION_THRESHOLD_DEG)
s = result['summary']
print('Sequence:', s['sequence_id'], '| Frames:', s['frame_count'], '| Pairs:', s['pair_count'])
print('Pose source:', s['pose_source'], '| Image sizes (W,H):', s['image_sizes_wh'])
print('Retained path length (m):', round(s['retained_path_length_m'], 4))
print('Start-to-end distance (m):', round(s['start_to_end_distance_m'], 4))
print('XYZ extent (m):', np.round(s['xyz_extent_m'], 4))
print('Source-ID gaps:', s['source_id_gaps'])
print('Pairs with translation <= 1 cm (or configured threshold):', s['low_translation_pair_count'])
print('Low translation + rotation above threshold:', s['low_translation_high_rotation_pair_count'])

statistics = pd.DataFrame({
    'translation_m': s['step_distance_m'],
    'rotation_deg': s['adjacent_rotation_deg'],
    'world_z_m': s['world_z_m'],
    'consecutive_ID_translation_m': s['consecutive_source_id_step_distance_m'],
    'consecutive_ID_rotation_deg': s['consecutive_source_id_rotation_deg'],
}).T
statistics['count'] = statistics['count'].astype(int)
display(statistics.round(5))


## Trajectory, altitude coordinate and motion distributions
نمودار مسیر از مرکز دوربین اول به‌عنوان مبدأ نمایشی استفاده می‌کند؛ محورهای جهان تغییر نمی‌کنند.
World Z ارتفاع در مختصات شهر است، نه ارتفاع از سطح زمین.
طول مسیر جمع پاره‌خط‌ها بین فریم‌های موجود است؛ مسیر طی‌شده بین عکس‌های حذف‌شده را بازسازی نمی‌کند.
نقاط نارنجی در نمودار حرکت، جفت‌هایی با شکاف شمارهٔ فریم هستند.


In [ ]:
figures = make_figures(result)
for name, figure in figures.items():
    display(figure)
    plt.close(figure)


## Rotation coverage
جدول زیر دامنهٔ pitch، yaw و roll اصلیِ CARLA را نشان می‌دهد.
`raw_span_deg` اختلاف بیشینه و کمینهٔ عددی است. `circular_span_deg` کوتاه‌ترین کمان
پوشانندهٔ زاویه‌هاست: برای 179 و -179 درجه، مقدار آن 2 درجه است، نه 358 درجه.

تغییر جهت بین تصاویر در نمودار حرکت با زاویهٔ ژئودزیک SO(3) محاسبه می‌شود و بین 0 و 180 درجه است.
دامنهٔ مؤلفه‌های Euler به قرارداد مختصات وابسته است؛ هنگام مقایسهٔ دیتاست‌ها قرارداد باید یکسان باشد.


In [ ]:
angle_ranges = pd.DataFrame(s['source_euler_ranges']).T
angle_ranges.index.name = 'source_CARLA_angle'
display(angle_ranges.round(5))


## Interactive 3D trajectory
می‌توانی مسیر را بچرخانی، زوم کنی و شمارهٔ فریم اصلی هر نقطه را ببینی.
اگر نمایش تعاملی در VS Code فعال نشد، فایل HTML ذخیره‌شده در سلول خروجی را در مرورگر باز کن.


In [ ]:
interactive = None
try:
    interactive = interactive_trajectory(result)
except ImportError:
    print('Optional interactive view: install plotly from requirements-analysis.txt.')
if interactive is not None:
    interactive.show()


## Inspect the largest frame-to-frame changes
این جفت‌ها صرفاً بیشترین تغییر را دارند؛ بزرگ بودن مقدار به‌تنهایی نشانهٔ خطا نیست.
برای بررسی تصویرِ یک جفت، شمارهٔ فریم جدید آن را در سلول پیش‌نمایش وارد کن.


In [ ]:
pairs = pd.DataFrame({
    'from_frame': np.arange(s['pair_count']),
    'to_frame': np.arange(1, s['frame_count']),
    'from_source_id': result['source_ids'][:-1],
    'to_source_id': result['source_ids'][1:],
    'source_id_delta': result['source_id_delta'],
    'distance_cm': 100 * result['step_distance_m'],
    'rotation_deg': result['rotation_step_deg'],
})
print('Largest translation steps:')
display(pairs.nlargest(10, 'distance_cm').round(5))
print('Largest orientation changes:')
display(pairs.nlargest(10, 'rotation_deg').round(5))


In [ ]:
from PIL import Image
PREVIEW_FRAMES = [0, s['frame_count']//2, s['frame_count']-1]
for index in dict.fromkeys(PREVIEW_FRAMES):
    row = result['dataset'].rows[index]
    print('Frame:', index, '| Source ID:', row['source_frame_id'], '| Source image:', row['source_image_name'])
    with Image.open(result['dataset'].paths[index]) as image:
        preview = image.convert('RGB')
        preview.thumbnail((1000, 500))
    display(preview)


## Compare sequences
مسیر مجموعه‌های دیگر را به فهرست اضافه کن. هر مجموعه مستقل تحلیل می‌شود و مسیر شهرهای مختلف
به هم متصل نمی‌شود. ورودی‌ها باید قالب Standard_GT داشته باشند؛ برای دیتاست خارجی ابتدا
واحدها و قرارداد پوز آن باید به همین قرارداد تبدیل شود.

برای مقایسهٔ منصفانه، نرخ نمونه‌برداری/انتخاب فریم و شکاف‌ها را هم در نظر بگیر.
بدون timestamp معتبر، سرعت، مدت ضبط یا FPS محاسبه نمی‌کنیم.


In [ ]:
COMPARE_SEQUENCES = [SEQUENCE]  # Add e.g. ROOT / 'standard_GT/city02_route01'
comparison_results = [result if Path(path).resolve() == SEQUENCE.resolve()
                      else analyze_sequence(path, TRANSLATION_THRESHOLD_M, ROTATION_THRESHOLD_DEG)
                      for path in COMPARE_SEQUENCES]
comparison = pd.DataFrame([comparison_row(item) for item in comparison_results])
display(comparison.round(4))


## Save tables and figures
یک پوشهٔ جدید برای این اجرا ساخته می‌شود: خلاصهٔ JSON، جدول مقایسه، آمار هر فریم و هر جفت،
نمودارهای PNG/PDF و مسیر تعاملی HTML. تصاویر اصلی دیتاست وارد گزارش نمی‌شوند.


In [ ]:
from datetime import datetime, timezone
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
REPORT_DIR = ROOT / 'outputs/dataset_stats' / s['sequence_id'] / RUN_TAG
export_statistics(result, REPORT_DIR)
statistics.to_csv(REPORT_DIR / 'distributions.csv', index_label='metric')
angle_ranges.to_csv(REPORT_DIR / 'angle_ranges.csv')
comparison.to_csv(REPORT_DIR / 'comparison.csv', index=False)
for name, figure in figures.items():
    figure.savefig(REPORT_DIR / (name + '.png'), dpi=180, bbox_inches='tight')
    figure.savefig(REPORT_DIR / (name + '.pdf'), bbox_inches='tight')
if interactive is not None:
    interactive.write_html(str(REPORT_DIR / 'trajectory_interactive.html'), include_plotlyjs=True)
print('Reports saved to:', REPORT_DIR)
